# 32 · Select rows by a range against pandas `between`

**Question.** Select rows by value keeps a range: from and to, both kept (Fieldwork experiment 171). It came from the
NYC smoke workflow, where fourteen days out of 3,653 were fourteen ticks. Does it keep exactly the rows
`Series.between(inclusive='both')` keeps?

The fixture is London's daily deaths 2002 to 2006, read from the published Stata file by Fieldwork's own reader.
Rows are compared date by date and in order, so the tolerance is **none**.


In [ ]:
# Only what this notebook imports. See 00-kernel-check for what this kernel really has.
NEEDED = ["pandas"]
try:
    import pyodide_js
    await pyodide_js.loadPackage(NEEDED)
    print("Requested:", NEEDED)
except ImportError:
    print("Not Pyodide; using the environment as installed.")


## Parity

Four cases: a range of dates (the August 2003 heatwave), a range of numbers (ozone 60 to 90), an open end, and the
number range again with one in-range cell emptied.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_32_select_range import run

report = run(Path.cwd() / "fixtures" / "select-range-london-001.json")
print("agrees:", report["agrees"])
print()
for c in report["comparisons"]:
    print(f"  {'OK ' if c['agrees'] else 'DIFF'} Fieldwork {c['fieldwork']:>4}  pandas {c['pandas']:>4}  {c['note']}")
    print(f"       {c['keyColumn']} from {c['from'] or 'the start'} to {c['to'] or 'the end'}: {c['first']} ... {c['last']}")


## The case worth the check: numbers are not text

Compared as text, `"100"` sorts before `"60"`, so a range of numbers keeps the wrong rows without any error. Both
implementations parse ozone as numbers; the cell below records what the text comparison would have kept instead.


In [ ]:
text = report["ozoneComparedAsText"]
print(f"ozone 60 to 90 as numbers : {text['insteadOf']} days")
print(f"ozone 60 to 90 as text    : {text['kept']} days   <- not what either implementation does")


## What this cannot establish

**That a range was the right selection.** A period chosen to show a peak is still a choice; the receipt records it
without judging it. **Datetime columns** are not exercised: the fixture's dates carry no time.


In [ ]:
for item in report["notEstablished"]:
    print("-", item)
print()
print("fixture :", report["fixture"]["id"], report["fixture"]["sha256"][:16])
print("pandas  :", report["environment"]["pandas"])
print("runtime :", report["environment"]["runtime"])
